# M04-3 Segmentacion

In [6]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)

ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated


In [7]:
from pyspark.sql.functions import col, sum as fsum, countDistinct, when, lit

spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
sales = fact.join(customers, "customer_id", "inner").where(col("is_billable"))
customer_gmv = sales.groupBy("customer_id", "country", "segment").agg(
    fsum("gmv_line").alias("gmv"),
    countDistinct("order_id").alias("orders"),
)
customer_gmv.orderBy(col("gmv").desc()).show(5)
print(customer_gmv.count())

+-----------+-------+---------+-----------------+------+
|customer_id|country|  segment|              gmv|orders|
+-----------+-------+---------+-----------------+------+
|      C0136|     IT| consumer|6007.196000000001|     5|
|      C0203|     PT|      vip|5904.559500000001|     5|
|      C0080|     IT|      vip|         5436.284|     5|
|      C0087|     FR| consumer|5151.370500000001|     3|
|      C0155|     IT|corporate|          5075.48|     6|
+-----------+-------+---------+-----------------+------+
only showing top 5 rows

211


## Paso 2 -- Bandas de negocio

In [8]:
banded = customer_gmv.withColumn(
    "value_band",
    when(col("gmv") < 1000, lit("low"))
    .when(col("gmv") < 3000, lit("mid"))
    .otherwise(lit("high")),
)
banded.groupBy("value_band").count().orderBy("value_band").show()


+----------+-----+
|value_band|count|
+----------+-----+
|      high|   40|
|       low|   56|
|       mid|  115|
+----------+-----+



## Paso 3 -- guarda para M05


In [9]:
banded.write.mode("overwrite").parquet(str(STAGING / "customer_gmv"))
print(spark.read.parquet(str(STAGING / "customer_gmv")).count())


211


## Mejora

In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import ntile

w = Window.orderBy(col("gmv"))
customer_gmv.withColumn("q", ntile(5).over(w)).groupBy("q").count().orderBy("q").show()


26/09/28 15:28:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 15:28:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 15:28:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 15:28:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 15:28:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 15:28:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 1

+---+-----+
|  q|count|
+---+-----+
|  1|   43|
|  2|   42|
|  3|   42|
|  4|   42|
|  5|   42|
+---+-----+

